# Tutorial 02: Using Real-World Maps with OSMnx

Learn how to create VRP instances from real-world street networks using OpenStreetMap data.

**What you'll learn:**
- Load street networks from OpenStreetMap using OSMnx
- Map real geographic coordinates to network nodes
- Compute network-based distances (actual driving distances)
- Create and solve PDPTW instances on real street networks

**Prerequisites:**
- Tutorial 01 (Quickstart)
- Basic understanding of geographic coordinates (latitude, longitude)

**Time:** ~20 minutes

**Note:** This tutorial requires the `osmnx` package. If not installed:
```bash
pip install osmnx
```

## 1. Setup and Imports

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import random

# VRP Toolkit OSMnx integration
from vrp_toolkit.data.osmnx_integration import (
    create_pdptw_from_osm,  # Complete workflow function
    OSMnxNetworkLoader,      # Manual network loading
    map_locations_to_nodes,  # Coordinate mapping
    compute_distance_matrix, # Distance computation
    compute_time_matrix      # Time computation
)

# PDPTW problem and solver
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import ALNSSolver, greedy_insertion_initial_solution

# Set random seed for reproducibility
seed_value = 42
np.random.seed(seed_value)
random.seed(seed_value)

print("Imports successful! OSMnx integration ready.")

## 2. Quick Start: Create PDPTW from Real Location

Let's start with the **simplest way** to create a PDPTW instance from real locations using the convenience function `create_pdptw_from_osm()`.

We'll use a small area in West Lafayette, IN (near Purdue University) as an example:

In [ ]:
# Define real geographic locations (latitude, longitude)
depot_location = (40.4237, -86.9212)  # Near Purdue Memorial Union

# Two pickup-delivery pairs
pickup_locations = [
    (40.4280, -86.9145),  # North campus
    (40.4200, -86.9180)   # South campus
]

delivery_locations = [
    (40.4250, -86.9100),  # East delivery
    (40.4210, -86.9220)   # West delivery
]

# Create PDPTW instance from OpenStreetMap
# This function does everything: loads network, maps nodes, computes matrices
order_table, distance_matrix, time_matrix, G, node_mapping = create_pdptw_from_osm(
    place_name="West Lafayette, Indiana, USA",
    depot_location=depot_location,
    pickup_locations=pickup_locations,
    delivery_locations=delivery_locations,
    cache_file="west_lafayette_network.graphml"  # Cache for faster subsequent loads
)

print(f"\nOrder table preview:")
print(order_table[['ID', 'Type', 'X', 'Y', 'RealIndex']].head())

**What just happened:**
1. We specified real GPS coordinates (latitude, longitude)
2. `create_pdptw_from_osm()` downloaded the street network for West Lafayette
3. It mapped our coordinates to the nearest street network nodes
4. It computed shortest path distances on the actual road network
5. It created a PDPTW-compatible order table

The result is a complete PDPTW instance using **real driving distances**, not straight-line Euclidean distances!

## 3. Create PDPTWInstance and Solve

Now let's create a PDPTWInstance and solve it using ALNS:

In [ ]:
# Create PDPTW instance
instance = PDPTWInstance(
    order_table=order_table,
    distance_matrix=distance_matrix,
    time_matrix=time_matrix,
    robot_speed=1.0  # Speed already factored into time_matrix
)

print(f"PDPTWInstance created:")
print(f"  - Orders: {instance.n}")
print(f"  - Total nodes: {len(instance.indices)}")
print(f"  - Average distance: {distance_matrix[distance_matrix > 0].mean():.2f} meters")
print(f"  - Average travel time: {time_matrix[time_matrix > 0].mean():.2f} minutes")

# Problem parameters
num_vehicles = 2
vehicle_capacity = 20
battery_capacity = 300
battery_consume_rate = 1
penalty_unvisit = 1000
penalty_delay = 50

# Generate initial solution
initial_solution = greedy_insertion_initial_solution(
    problem=instance,
    num_vehicles=num_vehicles,
    vehicle_capacity=vehicle_capacity,
    battery_capacity=battery_capacity,
    battery_consume_rate=battery_consume_rate,
    penalty_unvisit=penalty_unvisit,
    penalty_delay=penalty_delay
)

print(f"\nInitial solution:")
print(f"  - Objective: {initial_solution.objective_function():.2f}")
print(f"  - Feasible: {initial_solution.is_feasible()}")

In [ ]:
# Solve with ALNS
solver = ALNSSolver(config={
    'max_iterations': 50,
    'segment_length': 10,
    'num_segments': 5,
    'start_temp': 10000,
    'cooling_rate': 0.99,
    'num_removal': 1  # Small instance, remove 1 order at a time
})

print("Running ALNS optimization on real street network...")
best_solution = solver.solve(
    problem=instance,
    num_vehicles=num_vehicles,
    vehicle_capacity=vehicle_capacity,
    battery_capacity=battery_capacity,
    battery_consume_rate=battery_consume_rate,
    penalty_unvisit=penalty_unvisit,
    penalty_delay=penalty_delay
)

print(f"\nOptimization complete!")
print(f"  - Best objective: {best_solution.objective_function():.2f}")
print(f"  - Improvement: {initial_solution.objective_function() - best_solution.objective_function():.2f}")
print(f"  - Solution feasible: {best_solution.is_feasible()}")

## 4. Understanding the Components

Let's break down what happened behind the scenes:

### 4.1 Street Network Loading

OSMnx downloads street network data from OpenStreetMap:

In [ ]:
# Inspect the street network graph
print(f"Street Network Information:")
print(f"  - Nodes (intersections): {len(G.nodes)}")
print(f"  - Edges (road segments): {len(G.edges)}")
print(f"  - Network type: drive (for vehicles)")

# Sample a few nodes
sample_nodes = list(G.nodes(data=True))[:3]
print(f"\nSample nodes (OSM node data):")
for node_id, data in sample_nodes:
    print(f"  Node {node_id}: lat={data['y']:.6f}, lon={data['x']:.6f}")

### 4.2 Coordinate to Node Mapping

Our GPS coordinates are mapped to the **nearest street intersection**:

In [ ]:
# Show how VRP node IDs map to OSM node IDs
print("Node Mapping (VRP ID -> OSM Node ID):")
for vrp_id, osm_node in node_mapping.items():
    node_type = order_table[order_table['ID'] == vrp_id]['Type'].values[0]
    print(f"  VRP Node {vrp_id} ({node_type}) -> OSM Node {osm_node}")

print("\nThis mapping allows us to:")
print("  1. Use sequential VRP node IDs (0, 1, 2, ...) for algorithms")
print("  2. Keep track of actual OSM node IDs (large integers)")
print("  3. Compute distances on the actual street network")

### 4.3 Network Distance vs Euclidean Distance

Let's compare network-based distances with straight-line Euclidean distances:

In [ ]:
# Compute Euclidean distances from coordinates
def euclidean_distance(coord1, coord2):
    """Approximate Euclidean distance in meters using lat/lon."""
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    # Rough approximation: 1 degree ≈ 111km
    dx = (lon2 - lon1) * 111000 * np.cos(np.radians(lat1))
    dy = (lat2 - lat1) * 111000
    return np.sqrt(dx**2 + dy**2)

# Compare depot to first pickup
network_dist = distance_matrix[0, 1]  # Depot to first pickup (network distance)
euclidean_dist = euclidean_distance(depot_location, pickup_locations[0])

print(f"Distance from Depot to Pickup 1:")
print(f"  - Network distance (actual roads): {network_dist:.2f} meters")
print(f"  - Euclidean distance (straight line): {euclidean_dist:.2f} meters")
print(f"  - Ratio: {network_dist / euclidean_dist:.2f}x")
print(f"\nNetwork distances are typically 1.2-1.5x longer due to:")
print(f"  - Following actual street layouts")
print(f"  - One-way streets and turn restrictions")
print(f"  - Building obstructions")

## 5. Manual Workflow (Step-by-Step)

If you need more control, you can execute each step manually:

### 5.1 Load Network Manually

In [ ]:
# Load network step-by-step
loader = OSMnxNetworkLoader()

# Option 1: Load by place name
G_manual = loader.load_by_place(
    "West Lafayette, Indiana, USA",
    network_type='drive',
    cache_file="west_lafayette_network.graphml"
)

# Ensure connectivity (keep largest connected component)
G_manual = loader.ensure_connectivity()

print(f"Network loaded: {len(G_manual.nodes)} nodes, {len(G_manual.edges)} edges")

### 5.2 Load by Bounding Box (Alternative)

For precise geographic boundaries:

In [ ]:
# Load by bounding box (north, south, east, west)
loader_bbox = OSMnxNetworkLoader()
G_bbox = loader_bbox.load_by_bbox(
    north=40.4350,
    south=40.4150,
    east=-86.9000,
    west=-86.9300,
    network_type='drive'
)

print(f"Bounding box network: {len(G_bbox.nodes)} nodes, {len(G_bbox.edges)} edges")
print(f"\nUse bounding box when:")
print(f"  - You need precise geographic boundaries")
print(f"  - Place name query returns too large an area")
print(f"  - You want to control network size")

## 6. Advanced: Custom Parameters

### 6.1 Custom Time Windows and Service Times

In [ ]:
# Create instance with custom parameters
order_table_custom, dist_matrix_custom, time_matrix_custom, G_custom, node_map_custom = create_pdptw_from_osm(
    place_name="West Lafayette, Indiana, USA",
    depot_location=depot_location,
    pickup_locations=pickup_locations,
    delivery_locations=delivery_locations,
    cache_file="west_lafayette_network.graphml",
    # Custom parameters
    demand=15.0,  # Each order demand
    time_window=(0.0, 600.0),  # 10-hour time window
    service_time=10.0  # 10-minute service time
)

print("Custom instance created with:")
print(f"  - Demand per order: 15.0 units")
print(f"  - Time window: 0-600 minutes (10 hours)")
print(f"  - Service time: 10 minutes per stop")

### 6.2 Custom Travel Speed

In [ ]:
# Recompute time matrix with different speed
# Default speed: 30 km/h
time_matrix_30 = compute_time_matrix(distance_matrix, average_speed_kmh=30)

# Slower urban speed: 20 km/h
time_matrix_20 = compute_time_matrix(distance_matrix, average_speed_kmh=20)

# Fast highway speed: 50 km/h
time_matrix_50 = compute_time_matrix(distance_matrix, average_speed_kmh=50)

print("Average travel times:")
print(f"  - 20 km/h (urban): {time_matrix_20[time_matrix_20 > 0].mean():.2f} min")
print(f"  - 30 km/h (default): {time_matrix_30[time_matrix_30 > 0].mean():.2f} min")
print(f"  - 50 km/h (highway): {time_matrix_50[time_matrix_50 > 0].mean():.2f} min")

print("\nChoose speed based on:")
print("  - Urban areas: 20-30 km/h")
print("  - Suburban: 30-40 km/h")
print("  - Highway: 50-80 km/h")

## 7. Comparison and Best Practices

**When to use real street networks (OSMnx):**
- Planning actual delivery routes in cities
- When network distance differs significantly from Euclidean
- Realistic optimization for deployment
- Visualization on actual maps

**When to use synthetic data (RealMap):**
- Algorithm development and testing
- Rapid prototyping
- When network details don't matter
- Faster computation for large experiments

**Common pitfalls:**
1. **Large networks are slow** - Limit area size or use bounding boxes
2. **Cache networks** - Always use `cache_file` parameter to avoid re-downloading
3. **Coordinate order** - Remember: (latitude, longitude), not (x, y)
4. **Disconnected graphs** - Use `ensure_connectivity()` to avoid unreachable nodes
5. **Speed estimation** - Choose realistic speeds for your vehicle type and area

## 8. Summary

**What you learned:**
- ✅ Load real street networks from OpenStreetMap using OSMnx
- ✅ Map GPS coordinates to network nodes
- ✅ Compute network-based distances (actual roads, not straight lines)
- ✅ Create and solve PDPTW instances on real street networks
- ✅ Customize parameters (speed, time windows, service times)

**Key takeaways:**
1. Use `create_pdptw_from_osm()` for quick setup with real locations
2. Network distances are 1.2-1.5x longer than Euclidean distances
3. Always cache networks to avoid repeated downloads
4. Choose appropriate travel speeds for your scenario

**Next steps:**
- Try Tutorial 03 for custom problem creation
- Try Tutorial 07 for data generation workflows
- Experiment with your own city or campus!
- Add charging stations with `charging_location` parameter